In [1]:
from pathlib import Path
import csv
import json

from mido import MidiFile
from smile_to_midi_v10 import smiles_to_midi

SAMPLES_CSV = Path("../../apps/api/sample-molecule-midi/samples.csv").resolve()
OUT_DIR = Path("../../apps/api/sample-molecule-midi").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)


def drop_environment_track(midi_path: Path) -> bool:
    """删除第4轨（环境轨），若不存在则跳过。"""
    midi = MidiFile(str(midi_path))
    if len(midi.tracks) < 4:
        return False
    del midi.tracks[3]
    midi.save(str(midi_path))
    return True


with SAMPLES_CSV.open("r", encoding="utf-8", newline="") as f:
    all_samples = list(csv.DictReader(f))

samples = all_samples[:8]
print(f"读取示例分子: {len(samples)} 条")
print(f"输出目录: {OUT_DIR}")

In [2]:
generated = []

for row in samples:
    out_path = OUT_DIR / row["midi_file"]
    result_path = Path(
        smiles_to_midi(
            smiles=row["smiles"],
            output_path=str(out_path),
            title=row["name_en"],
            tempo_bpm=100,
            note_duration=0.8,
            overlap=0.7,
            rest_duration=0.1,
        )
    ).resolve()

    before_tracks = len(MidiFile(str(result_path)).tracks)
    removed = drop_environment_track(result_path)
    after_tracks = len(MidiFile(str(result_path)).tracks)

    generated.append(
        {
            "index": int(row["index"]),
            "id": row["id"],
            "name_zh": row["name_zh"],
            "name_en": row["name_en"],
            "midi_path": str(result_path),
            "tracks_before": before_tracks,
            "tracks_after": after_tracks,
            "environment_track_removed": removed,
        }
    )

print(json.dumps(generated, ensure_ascii=False, indent=2))

[MusicMol] Developed by:
 - Wanxiang Shen (PI)
 - Chao Cui
 - Jiayi Tang
 - Ziyan Zhu
 - Nan Ke


'/mnt/shenwanxiang/Research/musicmol_vs/model/分子到音乐/aspirin.mid'

In [3]:
# 快速确认 8 个 MIDI 文件都已写出
midi_files = sorted(str(p) for p in OUT_DIR.glob("*.mid") if p.name[:2].isdigit())
len(midi_files), midi_files

<music21.stream.Score 0x7f40e653dbd0>

In [ ]:
# 查看每个 MIDI 的轨道数，确认第4轨已移除（通常从4变3）
track_report = {
    item["id"]: len(MidiFile(item["midi_path"]).tracks)
    for item in generated
}
track_report

In [ ]:
# 简要结果（id -> 文件）
{item["id"]: item["midi_path"] for item in generated}

In [ ]:
### 说明

- 该 notebook 仅使用 `smile_to_midi_v10.smiles_to_midi(...)`。
- 输入分子来自 `../../apps/api/sample-molecule-midi/samples.csv` 的前 8 条。
- 每个 MIDI 生成后会删除第 4 轨（环境轨）。
- 依赖（如 `mido`, `music21`, `rdkit`）请按你的环境自行安装。